In [0]:
%run "./Util"

In [0]:
# ── CELL 1 : Imports + read silver ───────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

silver = spark.sql(f" SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")


In [0]:
def get_merge_stats(DIM_TABLE):
    return spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.{DIM_TABLE} ") \
        .select("*") \
        .orderBy(F.col("version").desc()) \
        .limit(1) \
        .withColumn("Status",
        F.when(F.col("operationMetrics.numSourceRows") == 0, "No Data to Load")
        .when(F.col("operationMetrics.numTargetRowsInserted") != 0, "Data Inserted")
        .when(
            (F.col("operationMetrics.numSourceRows") != 0) & (F.col("operationMetrics.numTargetRowsInserted") == 0),
            "No Change in Data"
        ).otherwise("Failure"))\
        .selectExpr("timestamp", "job", "operation", "operationMetrics.numOutputRows", "operationMetrics.numSourceRows", "operationMetrics.numTargetRowsInserted", "Status")


def quarantine_fact_inspection(quarantine_df):
    quarantine_df.createOrReplaceTempView("staging_quarantine_inspection")
    spark.sql(f""" 
        INSERT INTO {CATALOG}.{GOLD_SCHEMA}.{GOLD_QUARANTINE_INSPECTION}
        SELECT
            inspection_id, 
            restaurant_id,
            restaurant_name, 
            aka_name,  
            CAST(inspection_date AS DATE), 
            inspection_type, 
            inspection_result,
            violation_score,
            violation_code,
            violation_points,
            pass_fail_flag, 
            source_city, 
            address, 
            zip_code,
            quarantine_category,
            failed_column,
            source_system,
            pipeline_name,
            current_timestamp() AS _date_to_warehouse,
            current_timestamp()AS quarantine_timestamp
        FROM staging_quarantine_inspection
    """)

## Load Dim Date

In [0]:
# ── dim_date ────────────────────────────────────────

dim_date = (
    silver
    .select("inspection_date")
    .distinct()
    .filter(F.col("inspection_date").isNotNull())
    .withColumn("date_key",           F.date_format("inspection_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("full_date",          F.col("inspection_date"))
    .withColumn("year",               F.year("inspection_date"))
    .withColumn("month_name",         F.date_format("inspection_date", "MMMM"))
    .withColumn("month_number",       F.month("inspection_date"))
    .withColumn("day",                F.date_format("inspection_date", "EEEE"))
    .withColumn("quarter",            F.concat(F.lit("Q"), F.quarter("inspection_date")))
    .withColumn("week_number",        F.weekofyear("inspection_date"))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "date_key", "full_date", "year", "month_name",
        "month_number", "day", "quarter", "week_number",
        "_date_to_warehouse"
    )
)

dim_date.createOrReplaceTempView("staging_dim_date")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} AS target
    USING staging_dim_date AS source
    ON target.date_key = source.date_key
    WHEN NOT MATCHED THEN INSERT *
""")

# dim_date.write.format("delta").mode("append")\
#     .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_DATE}")

get_merge_stats(DIM_DATE).display()


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-12T23:08:36.000Z,null,MERGE,4519,4519,4519,Data Inserted


## Load Dim Violation Detail

In [0]:
# ── dim_violation_detail ───────────────────────────────────

dim_violation = (
    silver
    .select("violation_id", "violation_code", "violation_description", "violation_detail")
    .distinct()
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "violation_id", "violation_code",
        "violation_description", "violation_detail", "_date_to_warehouse"
    )
)

dim_violation.createOrReplaceTempView("staging_dim_violation_detail")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} AS target
    USING staging_dim_violation_detail AS source
    ON target.violation_id = source.violation_id
    WHEN NOT MATCHED THEN INSERT (
        violation_id, violation_code, violation_description, violation_detail, _date_to_warehouse
    )
    VALUES (
        source.violation_id, source.violation_code, source.violation_description, source.violation_detail, source._date_to_warehouse
    )
""")


get_merge_stats(DIM_VIOLATION_DETAIL).display()

timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-12T23:08:50.000Z,null,MERGE,911,911,911,Data Inserted


## Load Dim Restaurant

In [0]:
# ── dim_restaurant ───────────────────────────────────

dim_restaurant = (
    silver
    .select("restaurant_id", "dba_name", "aka_name", "facility_type", "address", "source_city", "state", "zip_code", "change_hash")
    .distinct()
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .alias("s")
    .join(
        spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
        .filter("is_current = true")
        .select("restaurant_id", "change_hash")
        .alias("t"),
        on=[
            F.col("s.restaurant_id") == F.col("t.restaurant_id"),
            F.col("s.change_hash") == F.col("t.change_hash")
        ],
        how="left_anti"
    )
)
dim_restaurant.createOrReplaceTempView("staging_dim_restaurant")


In [0]:
%sql



MERGE INTO final_project.gold_zone.dim_restaurant AS target
USING (
  SELECT restaurant_id AS merge_key, * FROM staging_dim_restaurant
  UNION ALL
  SELECT NULL AS merge_key, * FROM staging_dim_restaurant
) AS source
ON target.restaurant_id = source.merge_key
AND target.is_current = true

-- Expire old
WHEN MATCHED AND target.change_hash <> source.change_hash THEN
  UPDATE SET
    target.is_current = false,
    target.effective_end_date = current_timestamp()

-- Insert ONLY from NULL branch
WHEN NOT MATCHED AND source.merge_key IS NULL THEN
  INSERT (
    restaurant_id,
    dba_name,
    aka_name,
    facility_type,
    address,
    city,
    state,
    zip_code,
    change_hash,
    effective_start_date,
    effective_end_date,
    is_current,
    _date_to_warehouse
  )
  VALUES (
    source.restaurant_id,
    source.dba_name,
    source.aka_name,
    source.facility_type,
    source.address,
    source.source_city,
    source.state,
    source.zip_code,
    source.change_hash,
    current_timestamp(),
    TIMESTAMP '9999-12-31',
    true,
    source._date_to_warehouse
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
43686,0,0,43686


## Quarantine Duplicate Inspection if any

In [0]:
# quarantine_df = (silver_df.alias("s").join(
#     spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}").alias("t"),
#     F.col("s.inspection_id") == F.col("t.inspection_id"),
#     "inner"
#     ).select("s.*")
#     .withColumn("quarantine_category", F.lit("Duplicates"))
#     .withColumn("failed_column", F.lit("inspection_id"))
#     .withColumn("source_system", F.lit("silver"))
#     .withColumn("pipeline_name", F.lit("gold"))
# )
# quarantine_fact_inspection(quarantine_df)

## Load Fact Inspection

In [0]:
# ── Fact Inspection ───────────────────────────────────

dim_date_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_DATE}")
    .select("date_key", "full_date")
)

w = Window.partitionBy("restaurant_id").orderBy(F.col("restaurant_key").desc())

dim_rest_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
    # .filter(F.col("is_current") == True)
    .select("restaurant_key", "restaurant_id", "dba_name", "aka_name", "facility_type", "change_hash")
    .distinct()
    # .withColumn("rn", F.row_number().over(w))
    # .filter(F.col("rn") == 1)
    # .drop("rn")
)

silver_df = silver.alias("s")
dim_rest_lkp = dim_rest_lkp.alias("r")
dim_date_lkp = dim_date_lkp.alias("dt")


silver_df_agg = (
    silver
    .groupBy("inspection_id", "inspection_date", "restaurant_id")
    .agg(F.countDistinct("violation_id").alias("total_violation").cast(IntegerType()),
        F.sum(F.coalesce(F.col("violation_points").cast(IntegerType()), F.lit(0))).alias("total_violation_points")
    )
).alias("agg")

fact_inspections = (
    silver_df_agg
    .join(silver.alias("s"), on=["inspection_id", "inspection_date", "restaurant_id"], how="left")
    .select("s.inspection_id", "s.restaurant_id", "dba_name", "aka_name", "source_city", "zip_code",
        "s.inspection_date", "inspection_type", "risk_category", "facility_type", "address", "license_num",
        "zip_code", "violation_score", "inspection_result", "pass_fail_flag", "agg.total_violation", "agg.total_violation_points"
    )
    .distinct()

    # Resolve date key
    .join(dim_date_lkp, F.col("s.inspection_date") == F.col("dt.full_date"), "left")

    # Resolve restaurant key
    .join(dim_rest_lkp,
          ((F.col("s.restaurant_id")    == F.col("r.restaurant_id"))   &
            (F.col("s.dba_name")        == F.col("r.dba_name"))        &
            (F.col("s.aka_name")        == F.col("r.aka_name"))
          ), "left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_id", "restaurant_key", F.col("license_num").cast(IntegerType()), "risk_category", F.col("date_key").alias("inspection_date_key"), F.col("violation_score").alias("inspection_score"), "inspection_result", 
        F.col("total_violation").cast(IntegerType()), F.col("total_violation_points").cast(IntegerType()), 
        "inspection_type", "source_city", "pass_fail_flag", "_date_to_warehouse"
    )
)

fact_inspections.createOrReplaceTempView("staging_fact_inspections")

# Append Only
spark.sql(f""" 
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS} target
    USING staging_fact_inspections source
    ON target.inspection_id = source.inspection_id
    -- Safe Append 
    WHEN NOT MATCHED THEN INSERT ( 
        inspection_id, restaurant_key, license_num, risk_category, inspection_date_key, inspection_score, inspection_result, total_violation, total_violation_points, inspection_type, source_city, pass_fail_flag, _date_to_warehouse
    )
    Values (
        source.inspection_id, source.restaurant_key, source.license_num, source.risk_category, source.inspection_date_key, source.inspection_score, source.inspection_result, source.total_violation, source.total_violation_points, source.inspection_type, source.source_city, source.pass_fail_flag, source._date_to_warehouse
    )
""")

get_merge_stats(FACT_INSPECTIONS).display()


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-12T23:09:31.000Z,null,MERGE,279611,279611,279611,Data Inserted


## Load Fact Violations

In [0]:

dim_viol_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL}")
    .select("violation_code_key", "violation_id")
)

w = Window.partitionBy("inspection_id").orderBy(F.col("inspection_score").desc())

fact_insp_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}")
    .select("inspection_key", "inspection_id", "inspection_score")
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("inspection_key", "inspection_id")
)

fact_violations = (
    silver
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    
    # Resolve violation_code_key from dim_violation
    .join(dim_viol_lkp,
          on=["violation_id"],
          how="left")
    # Resolve inspection_key from fact_inspections
    .join(fact_insp_lkp,
          on="inspection_id",
          how="left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_key",
        "violation_code_key",
        F.col("violation_points"),
        F.col("is_critical_flag").alias("is_critical"),
        F.col("inspector_comments").alias("violation_comment"),
        "_date_to_warehouse"
    )
)

fact_violations.createOrReplaceTempView("staging_fact_violations")

# Append Only
spark.sql(f""" 
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_VIOLATIONS} target
    USING staging_fact_violations source
    ON target.inspection_key = source.inspection_key
    AND target.violation_code_key = source.violation_code_key

    -- Safe Append 
    WHEN NOT MATCHED THEN INSERT ( 
        inspection_key, violation_code_key, violation_points, is_critical, violation_comment, _date_to_warehouse
    )
    Values (
        source.inspection_key, source.violation_code_key, source.violation_points, source.is_critical, source.violation_comment, source._date_to_warehouse
    )
""")

get_merge_stats(FACT_VIOLATIONS).display()

timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-12T23:09:43.000Z,null,MERGE,1179883,1179883,1179883,Data Inserted
